# NDT7 (M-Lab) Data Prep — Vietnam Broadband + Mobile, Province x Quarter

Aggregates `../../../data/ndt7/vn/mlab_vn_clean.parquet` (24.7M raw NDT7 test records, already ISP-classified and province-joined via
per-IP lookup + point-in-polygon) into province x quarter format, split into Broadband and
Mobile/Cellular parts, mirroring the same structure across all three NDT7 "tigger" countries
(Cambodia/Thailand/Vietnam).

**Rebuilt to use DuckDB instead of a manual pyarrow-batch-streaming loop** — DuckDB reads the
parquet file directly and does the tile-binning + GROUP BY aggregation out-of-core (no manual
batching code needed, no risk of the memory issues the streaming version was written to avoid).
The tile-binning and weighted-aggregation formulas are byte-for-byte unchanged from the pandas
version — verified against the prior pandas-based export (float-precision-only differences,
~1e-13, from AVG() accumulation order).

No province-name mapping needed — Vietnam's raw `province` values already match `vietnam_reference.csv`.

Same tile scheme as Ookla's own published tiles (zoom-16 slippy tiles, ~610m) — keeps
`n_tiles`/`is_reliable` comparable across Ookla and NDT7, and across countries:
`total_tests >= 100` only (n_tiles dropped for NDT7 — MaxMind gives city-centroid coordinates, so n_tiles measures cities-per-province, not data spread; Ookla keeps both).

**Outputs:**
- `data/exports/ndt7_vietnam_province_quarterly.csv` — Broadband
- `data/exports/ndt7_mobile_vietnam_province_quarterly.csv` — Mobile/Cellular
  (renamed from `ndt7_vietnam_mobile_...` to match Ookla's `ookla_mobile_<country>_...`
  naming convention — position of "mobile" now matches across both pipelines)

In [1]:
import duckdb
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

RAW_PARQUET = '../../../data/ndt7/vn/mlab_vn_clean.parquet'
VN_REF_CSV = '../../../data/reference/vietnam_reference.csv'

ZOOM = 16
N_TILES = 2 ** ZOOM
MIN_TILE_TESTS = 3

### 1. Tile-Binning + Province-Quarter Aggregation (DuckDB)

All heavy row-level work (filtering, quarter-labeling, zoom-16 mercator tile assignment, GROUP BY tile x quarter x type x network_type) happens in one DuckDB SQL query against the raw parquet — no Python-side batching.

In [2]:
con = duckdb.connect()
con.execute("SET memory_limit='3GB'")                        # PH/ID ใหญ่ ต้องตั้ง
con.execute("SET temp_directory='../../../.tmp/duckdb'")   # ที่พักตอน spill
con.execute("SET preserve_insertion_order=false")

sql = f"""
WITH filtered AS (
    SELECT
        mean_throughput_mbps,
        min_rtt,
        type, network_type, province,
        year,
        CAST(CEIL(month / 3.0) AS INT) AS qtr,
        -- zoom-16 Web Mercator tile — ใช้เป็นคอลัมน์วินิจฉัยเท่านั้น ไม่ได้ใช้คิดค่าเฉลี่ย
        CAST(FLOOR((longitude + 180) / 360 * 65536) AS BIGINT) AS tx,
        CAST(FLOOR((1 - (ln(tan(radians(LEAST(GREATEST(latitude, -85.05112878), 85.05112878)))
             + 1.0/cos(radians(LEAST(GREATEST(latitude, -85.05112878), 85.05112878))))) / pi()) / 2 * 65536) AS BIGINT) AS ty
    FROM read_parquet('{RAW_PARQUET}')
    WHERE mean_throughput_mbps > 0
      AND province IS NOT NULL
      AND network_type IN ('broadband', 'cellular')
)
SELECT
    province, network_type, type,
    (CAST(year AS VARCHAR) || '-Q' || CAST(qtr AS VARCHAR)) AS year_q,
    AVG(mean_throughput_mbps)                       AS avg_thr,
    AVG(CASE WHEN min_rtt < 2000 THEN min_rtt END)  AS avg_lat,
    COUNT(*)                                        AS test_count,
    COUNT(DISTINCT tx * 65536 + ty) AS n_tiles
FROM filtered
GROUP BY province, network_type, type, year_q
"""

tile_agg_all = con.execute(sql).df()
print(f"province x quarter x type x network rows: {len(tile_agg_all):,}")
print(f"quarters: {len(tile_agg_all['year_q'].unique())} | province: {tile_agg_all['province'].nunique()}")
print(tile_agg_all.groupby('network_type')['test_count'].sum().apply(lambda x: f'{x:,}'))

province x quarter x type x network rows: 2,103
quarters: 12 | province: 63
network_type
broadband    21,817,334
cellular      1,047,714
Name: test_count, dtype: str


### 3. Province-Level Weighted Aggregation (per network type)

In [3]:
def build_province_quarterly(tile_agg_all, network_type, ref):
    d = tile_agg_all[tile_agg_all['network_type'] == network_type]
    print(f"[{network_type}] province x quarter x type rows: {len(d):,}")

    dl = d[d['type'] == 'download'].rename(columns={
        'avg_thr': 'avg_d_mbps', 'avg_lat': 'avg_lat_ms_wt', 'test_count': 'total_tests'})
    ul = d[d['type'] == 'upload'].rename(columns={'avg_thr': 'avg_u_mbps'})

    dl_stats = dl[['year_q', 'province', 'avg_d_mbps', 'avg_lat_ms_wt', 'total_tests', 'n_tiles']]
    ul_stats = ul[['year_q', 'province', 'avg_u_mbps']]

    master = pd.merge(dl_stats, ul_stats, on=['year_q', 'province'], how='outer')
    master = master.rename(columns={'year_q': 'quarter'})
    master['year'] = master['quarter'].str.slice(0, 4).astype(int)
    master['quarter.1'] = master['quarter'].str.slice(6, 7).astype(int)

    # NDT7 ใช้ total_tests อย่างเดียว ไม่ใช้ n_tiles เป็นเกณฑ์ (Ookla ยังใช้ทั้งคู่)
    # เหตุผล: NDT7 ได้พิกัดจาก MaxMind ซึ่งเป็น city centroid ทุก test ในเมืองเดียวกันจึงตกลง tile
    # เดียวกัน n_tiles จึงวัด "จังหวัดนี้มีกี่เมืองใน MaxMind" ไม่ได้วัดการกระจายตัวของข้อมูล
    # (ลาวทั้งประเทศมีพิกัดต่างกัน 33 จุด n_tiles สูงสุด = 3 -> เกณฑ์ >=5 เป็นไปไม่ได้)
    # คอลัมน์ n_tiles ยังเก็บไว้ให้ดูใน "Data Quality" ของ EDA
    master['is_reliable'] = master['total_tests'] >= 100
    print(f"[{network_type}] province x quarter rows: {len(master)} | "
          f"reliable: {master['is_reliable'].sum()} ({master['is_reliable'].mean():.1%})")

    master = master.merge(
        ref[['province_en', 'region', 'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021',
             'density_per_km2', 'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']],
        left_on='province', right_on='province_en', how='left'
    ).drop(columns=['province_en'])

    missing_ref = master[master['region'].isna()]['province'].unique()
    if len(missing_ref):
        print(f"[{network_type}] WARNING — no reference match: {list(missing_ref)}")

    return master


EXPORT_COLS = ['province', 'quarter', 'year', 'quarter.1', 'avg_d_mbps', 'avg_u_mbps',
               'avg_lat_ms_wt', 'total_tests', 'n_tiles', 'is_reliable', 'region',
               'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021', 'density_per_km2',
               'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']

In [4]:
ref = pd.read_csv(VN_REF_CSV)

---
## Part 1 — Broadband

In [5]:
broadband_master = build_province_quarterly(tile_agg_all, 'broadband', ref)
broadband_master.head()

[broadband] province x quarter x type rows: 1,492
[broadband] province x quarter rows: 746 | reliable: 698 (93.6%)
[broadband] WARNING — no reference match: ['AnGiang', 'BàRịa-VũngTàu', 'BìnhDương', 'BìnhPhước', 'BìnhThuận', 'BìnhĐịnh', 'BạcLiêu', 'BắcGiang', 'BắcKạn', 'BắcNinh', 'BếnTre', 'CaoBằng', 'CàMau', 'CầnThơ', 'GiaLai', 'HoàBình', 'HàGiang', 'HàNam', 'HàNội', 'HàTĩnh', 'HưngYên', 'HảiDương', 'HảiPhòng', 'HậuGiang', 'HồChíMinh', 'KhánhHòa', 'KiênGiang', 'KonTum', 'LaiChâu', 'LongAn', 'LàoCai', 'LâmĐồng', 'LạngSơn', 'NamĐịnh', 'NghệAn', 'NinhBình', 'NinhThuận', 'PhúThọ', 'PhúYên', 'QuảngBình', 'QuảngNam', 'QuảngNgãi', 'QuảngNinh', 'QuảngTrị', 'SócTrăng', 'SơnLa', 'ThanhHóa', 'TháiBình', 'TháiNguyên', 'ThừaThiênHuế', 'TiềnGiang', 'TràVinh', 'TuyênQuang', 'TâyNinh', 'VĩnhLong', 'VĩnhPhúc', 'YênBái', 'ĐiệnBiên', 'ĐàNẵng', 'ĐắkLắk', 'ĐắkNông', 'ĐồngNai', 'ĐồngTháp']


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,2023-Q1,AnGiang,28.380371,110.829711,779,13,31.584573,2023,1,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-Q1,BàRịa-VũngTàu,35.195669,113.342759,1570,9,28.492537,2023,1,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-Q1,BìnhDương,31.687039,108.778676,722,11,25.492240,2023,1,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-Q1,BìnhPhước,28.983535,124.874920,276,8,24.895259,2023,1,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023-Q1,BìnhThuận,31.661640,96.690196,321,7,29.740104,2023,1,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
out_bb = broadband_master[EXPORT_COLS].copy()
OUT_PATH_BB = '../../../data/exports/ndt7_vietnam_province_quarterly.csv'
out_bb.to_csv(OUT_PATH_BB, index=False)
print(f"Exported {len(out_bb)} rows -> {OUT_PATH_BB}")
out_bb.head(3)

Exported 746 rows -> ../../../data/exports/ndt7_vietnam_province_quarterly.csv


,province,quarter,year,quarter.1,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,AnGiang,2023-Q1,2023,1,28.380371,31.584573,110.829711,779,13,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BàRịa-VũngTàu,2023-Q1,2023,1,35.195669,28.492537,113.342759,1570,9,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BìnhDương,2023-Q1,2023,1,31.687039,25.492240,108.778676,722,11,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---
## Part 2 — Mobile/Cellular

In [7]:
mobile_master = build_province_quarterly(tile_agg_all, 'cellular', ref)
mobile_master.head()

[cellular] province x quarter x type rows: 611
[cellular] province x quarter rows: 308 | reliable: 81 (26.3%)
[cellular] WARNING — no reference match: ['AnGiang', 'BàRịa-VũngTàu', 'BìnhDương', 'BìnhPhước', 'BìnhThuận', 'BìnhĐịnh', 'BắcGiang', 'BắcKạn', 'BắcNinh', 'BếnTre', 'CàMau', 'CầnThơ', 'HoàBình', 'HàNội', 'HưngYên', 'HảiDương', 'HảiPhòng', 'HồChíMinh', 'KiênGiang', 'LongAn', 'NamĐịnh', 'NghệAn', 'NinhBình', 'PhúThọ', 'QuảngNam', 'QuảngTrị', 'SócTrăng', 'ThanhHóa', 'TháiNguyên', 'ThừaThiênHuế', 'TiềnGiang', 'TâyNinh', 'VĩnhLong', 'VĩnhPhúc', 'YênBái', 'ĐàNẵng', 'ĐắkNông', 'ĐồngNai', 'GiaLai', 'KhánhHòa', 'QuảngNgãi', 'QuảngNinh', 'SơnLa', 'TháiBình', 'ĐồngTháp', 'LạngSơn', 'PhúYên', 'ĐiệnBiên', 'HàTĩnh', 'LàoCai', 'HàGiang', 'HàNam', 'LaiChâu', 'KonTum', 'ĐắkLắk', 'TràVinh', 'LâmĐồng', 'TuyênQuang']


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,2023-Q1,AnGiang,NaN,NaN,NaN,NaN,6.886281,2023,1,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-Q1,BàRịa-VũngTàu,35.281630,160.264800,5.0,2.0,12.991238,2023,1,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-Q1,BìnhDương,17.188266,179.231250,8.0,1.0,6.633346,2023,1,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-Q1,BìnhPhước,40.413733,113.362200,5.0,1.0,15.769871,2023,1,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023-Q1,BìnhThuận,38.804946,121.847365,85.0,3.0,33.034556,2023,1,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
out_mb = mobile_master[EXPORT_COLS].copy()
OUT_PATH_MB = '../../../data/exports/ndt7_mobile_vietnam_province_quarterly.csv'
out_mb.to_csv(OUT_PATH_MB, index=False)
print(f"Exported {len(out_mb)} rows -> {OUT_PATH_MB}")
out_mb.head(3)

Exported 308 rows -> ../../../data/exports/ndt7_mobile_vietnam_province_quarterly.csv


,province,quarter,year,quarter.1,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,AnGiang,2023-Q1,2023,1,NaN,6.886281,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BàRịa-VũngTàu,2023-Q1,2023,1,35.281630,12.991238,160.26480,5.0,2.0,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BìnhDương,2023-Q1,2023,1,17.188266,6.633346,179.23125,8.0,1.0,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Summary

- Input: Vietnam NDT7 raw test records, already province-joined + ISP-classified
- Output: province x quarter aggregates for Broadband and Mobile separately, tile-binned at
  Ookla's zoom-16 resolution, same `is_reliable` threshold as every Ookla country notebook and
  the other NDT7 "tigger" prep notebooks
- Engine: DuckDB (was: manual pyarrow-batch-streaming loop in pandas) — verified to reproduce
  the prior pandas-based export exactly (float-precision-only differences)